# 02 — Préparation des données HTRogène

Extraction des lignes de texte des 19 manuscrits occitans du corpus [HTRogène](https://github.com/HTRogene/occitan) (CC-BY 4.0), à partir des fichiers ALTO normalisés (`*.chocomufin.xml`), puis découpage train / dev / test **par manuscrit**.

Deux modes de découpage des lignes :
- `"rectangle"` : rectangle englobant de la ligne (attributs `HPOS`, `VPOS`, `WIDTH`, `HEIGHT`) ;
- `"polygone"` : polygone de la ligne, avec tout ce qui sort du polygone masqué en blanc. **C'est le mode utilisé pour tous les résultats.** Le mode rectangle inclut les lignes voisines sur les pages photographiées de travers (CER de Kraken tel quel : 40,1 % contre 16,7 %).

**Sortie** : `ROOT/lignes_occitan_poly/` (ou `lignes_occitan/` en mode rectangle), avec un dossier `images/` et un fichier `lignes.jsonl` (une ligne JSON par ligne de manuscrit : `image`, `text`, `manuscrit`, `page`, `split`).

**Environnement** : Google Colab, sans GPU.

In [ ]:
MODE = "polygone"          # "polygone" ou "rectangle"

from pathlib import Path
try:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/projet_vlm")
except ImportError:
    ROOT = Path("projet_vlm")
ROOT.mkdir(parents=True, exist_ok=True)

!git clone -q --depth 1 https://github.com/HTRogene/occitan

In [ ]:
import json, shutil
import xml.etree.ElementTree as ET
from collections import Counter
from PIL import Image, ImageDraw

NS = {"a": "http://www.loc.gov/standards/alto/ns-v4#"}
SRC = Path("occitan/data")

# Découpage par manuscrit. paris-bnf-fr-13509 reste dans le train :
# il fait partie de CATMuS Medieval, sur lequel le modèle Kraken de départ a été entraîné.
TEST = {"paris-bnf-naf-6195", "additional-21218", "paris-bnf-arsenal-6355"}
DEV  = {"additional-10323", "paris-bnf-fr-13504", "london-bl-harley-3041"}

def decouper(page, ligne):
    if MODE == "rectangle":
        x, y = int(float(ligne.get("HPOS"))), int(float(ligne.get("VPOS")))
        l, h = int(float(ligne.get("WIDTH"))), int(float(ligne.get("HEIGHT")))
        return page.crop((x, y, x + l, y + h))
    c = [float(v) for v in ligne.find("a:Shape/a:Polygon", NS).get("POINTS").split()]
    points = list(zip(c[0::2], c[1::2]))
    x0, y0 = int(min(p[0] for p in points)), int(min(p[1] for p in points))
    x1, y1 = int(max(p[0] for p in points)) + 1, int(max(p[1] for p in points)) + 1
    decoupe = page.crop((x0, y0, x1, y1))
    masque = Image.new("L", decoupe.size, 0)
    ImageDraw.Draw(masque).polygon([(x - x0, y - y0) for x, y in points], fill=255)
    fond = Image.new("RGB", decoupe.size, (255, 255, 255))
    fond.paste(decoupe, mask=masque)
    return fond

In [ ]:
LOCAL = Path("lignes_occitan_poly" if MODE == "polygone" else "lignes_occitan")
(LOCAL / "images").mkdir(parents=True, exist_ok=True)

exemples, vides = [], 0
for xml_path in sorted(SRC.glob("*/*.chocomufin.xml")):
    manuscrit = xml_path.parent.name
    page_nom = xml_path.name.replace(".chocomufin.xml", "")
    img_path = xml_path.parent / f"{page_nom}.jpg"
    if not img_path.exists():          # METS.chocomufin.xml : un sommaire, pas une page
        continue
    page = Image.open(img_path).convert("RGB")

    for i, ligne in enumerate(ET.parse(xml_path).getroot().findall(".//a:TextLine", NS)):
        string = ligne.find("a:String", NS)
        texte = string.get("CONTENT", "").strip() if string is not None else ""
        if not texte or (MODE == "polygone" and ligne.find("a:Shape/a:Polygon", NS) is None):
            vides += 1
            continue
        nom = f"{manuscrit}__{page_nom}__{i:03d}.png"
        decouper(page, ligne).save(LOCAL / "images" / nom)
        split = "test" if manuscrit in TEST else "dev" if manuscrit in DEV else "train"
        exemples.append({"image": f"images/{nom}", "text": texte,
                         "manuscrit": manuscrit, "page": page_nom, "split": split})

with open(LOCAL / "lignes.jsonl", "w", encoding="utf-8") as f:
    for ex in exemples:
        f.write(json.dumps(ex, ensure_ascii=False) + "\n")

print(len(exemples), "lignes,", vides, "écartées", Counter(e["split"] for e in exemples))
# Attendu : 4293 lignes, 132 écartées, train 3392 / test 574 / dev 327
for m, n in Counter(e["manuscrit"] for e in exemples).most_common():
    print(f"{n:5}  {m}")

## Vérification visuelle, puis copie dans le dossier de travail

In [ ]:
from IPython.display import display
for e in [e for e in exemples if e["manuscrit"] == "paris-bnf-naf-6195"][:5]:
    display(Image.open(LOCAL / e["image"]))
    print(e["text"])

In [ ]:
shutil.copytree(LOCAL, ROOT / LOCAL.name, dirs_exist_ok=True)
print("Copié dans", ROOT / LOCAL.name)